# PG-LIF -- SSC (Spiking Speech Commands) Confirmation Run
**Purpose.** The single most-requested item across all seven pre-submission reviews (6 of 7 independently
flagged it): Section 6 promises SSC as part of the evaluation, but it was never run. This notebook runs it.

**Same model code, different dataset.** SSC uses the identical 700-channel cochlea-model input format as
SHD (same HDF5 layout: `spikes/times`, `spikes/units`, `labels`), so every neuron class, `TAG_SPEC` entry,
and the training loop itself are copied verbatim from the verified `PG_LIF_P1_paper_mode.ipynb` harness --
nothing about the model changes. Only the data pipeline and the epoch budget are adapted, for reasons
specific to SSC's scale, explained below.

**Two adaptations, and why each is necessary:**
1. **On-the-fly batch binning instead of the dense precompute SHD uses.** SHD's harness bins the entire
   dataset into one dense boolean tensor up front (fast, and SHD is small enough that this costs ~1.4 GB).
   SSC has 75,466 training samples versus SHD's 8,156 -- roughly 9.3x more -- so the same approach would
   need approximately 13 GB just for the training tensor, which risks out-of-memory failures on typical
   Colab instances. This notebook bins each batch on demand instead, trading some CPU time for a bounded,
   safe memory footprint.
2. **Reduced epoch budget.** SHD's protocol is 100 epochs. At SSC's per-epoch cost (~9.3x more samples,
   plus the on-the-fly binning overhead), 100 epochs would take roughly a day of wall-clock time per single
   (tag, seed) run -- impractical for a confirmation study. We use 15 epochs instead: 15 epochs x 75,466
   samples ≈ 1.13M sample-presentations, which is already *more* total gradient updates than SHD's own
   100 epochs x 8,156 samples ≈ 815K -- so this is not a weaker protocol, just a shorter one in epoch-count
   terms. Early stopping remains active and will cut this further if a model plateaus sooner.

**Default scope, deliberately conservative.** The five models of Table 1 (LIF, ALIF, TC-LIF, DH-LIF, PG-LIF)
at **1 seed** to start -- not the full ablation battery, and not 5 seeds, given the added per-run cost. This
is easy to widen later using the exact same resume/skip pattern already proven in the main harness: edit
`MODELS` / `SEEDS` below and re-run; anything already finished is skipped automatically.

In [1]:
import os, json, time, math
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/My Drive'
    if not os.path.isdir(ROOT): ROOT = '/content/drive/MyDrive'
    BASE = os.path.join(ROOT, 'PG_LIF')
except Exception:
    print('WARNING: not in Colab or Drive mount failed -- results will not persist.'); BASE = './PG_LIF'
DATA = os.path.join(BASE, 'data', 'SSC')
OUT = os.path.join(BASE, 'P1_results', 'paper_mode_ssc')   # OWN folder -- cannot collide with any SHD results
os.makedirs(DATA, exist_ok=True); os.makedirs(OUT, exist_ok=True)
print('Results folder:', OUT)

T_BINS, N_IN, N_OUT, HIDDEN = 250, 700, 35, 128   # N_OUT=35 (SSC classes; SHD uses 20). Everything else
                                                    # identical to the confirmed SHD protocol.
BATCH, LR, MAX_TIME = 64, 5e-4, 1.4
EPOCHS = 15         # see markdown above: this is MORE total gradient updates than SHD's 100-epoch protocol,
                    # not fewer, once SSC's larger per-epoch sample count is accounted for.
EARLY_STOP_PATIENCE, EARLY_STOP_MIN_EPOCH = 5, 5   # tightened from SHD's (15, 45): at 15 total epochs,
                                                    # SHD's patience=15 would never trigger at all.
DROPOUT = 0.1
USE_RECURRENCE = True   # matches the SHD harness's confirmed default (KEEP TRUE; parameter count match)
SCHEDULE = [6, 11]   # v9's [40,80] scaled to this notebook's 15-epoch budget in the same 40%/80% proportion
V_CLAMP, P_CLAMP = 20.0, 50.0

MODELS = ['LIF', 'ALIF', 'TCLIF', 'DHLIF', 'PGLIF']   # Table 1's five; widen to add ablations later if wanted
SEEDS = [0]                                            # widen to [0,1,2] etc. across sessions once affordable
print(f'This session will train/resume: {MODELS} x seeds {SEEDS}')
print('Estimate: each (tag, seed) run is a SEPARATE multi-hour job at SSC scale -- budget accordingly '
      'and use the per-(tag,seed) resume logic across sessions, exactly as with the SHD harness.')

Mounted at /content/drive
Results folder: /content/drive/My Drive/PG_LIF/P1_results/paper_mode_ssc
This session will train/resume: ['LIF', 'ALIF', 'TCLIF', 'DHLIF', 'PGLIF'] x seeds [0]
Estimate: each (tag, seed) run is a SEPARATE multi-hour job at SSC scale -- budget accordingly and use the per-(tag,seed) resume logic across sessions, exactly as with the SHD harness.


## Data: download and on-the-fly binning
Downloads `ssc_train.h5.gz` / `ssc_test.h5.gz` from the same host and URL pattern as SHD (confirmed to work
in the existing harness). We deliberately do not download or use `ssc_valid.h5`: the manuscript's SHD
protocol trains directly on the training split and evaluates on the test split with no validation-based
model selection, and we keep SSC's protocol consistent with that rather than introduce a methodological
difference between the two datasets within the same paper.

In [2]:
import numpy as np, h5py, torch, torch.nn as nn, gzip, shutil, urllib.request
URLS = {'ssc_train.h5': 'https://zenkelab.org/datasets/ssc_train.h5.gz',
        'ssc_test.h5':  'https://zenkelab.org/datasets/ssc_test.h5.gz'}
for name, url in URLS.items():
    dst = os.path.join(DATA, name)
    if not os.path.exists(dst):
        gz = dst + '.gz'
        print('downloading', url, '... (SSC is much larger than SHD -- this will take a while)')
        urllib.request.urlretrieve(url, gz)
        with gzip.open(gz, 'rb') as fi, open(dst, 'wb') as fo: shutil.copyfileobj(fi, fo)
        os.remove(gz)
    print(name, 'ready,', os.path.getsize(dst) // (1 << 20), 'MB, at', dst)

def load_split(fname):
    with h5py.File(os.path.join(DATA, fname), 'r') as f:
        return ([np.array(t) for t in f['spikes']['times']],
                [np.array(u) for u in f['spikes']['units']],
                np.array(f['labels'], dtype=np.int64))
TR_raw = load_split('ssc_train.h5'); TE_raw = load_split('ssc_test.h5')
print('train', len(TR_raw[2]), '| test', len(TE_raw[2]))

def bin_batch(times_list, units_list, labels_arr, idx):
    """Bin only the requested indices, on demand -- bounded memory regardless of dataset size. Uses the
    identical np.digitize(t, linspace(0,MAX_TIME,T_BINS)) scheme as the SHD harness, for fidelity."""
    time_bins = np.linspace(0, MAX_TIME, num=T_BINS)
    B = len(idx)
    X = torch.zeros(B, T_BINS, N_IN, dtype=torch.bool)
    for bi, i in enumerate(idx):
        tt = times_list[i]; uu = units_list[i]
        tb = np.clip(np.digitize(tt, time_bins), 0, T_BINS - 1)
        X[bi, tb, uu] = True
    return X, torch.as_tensor(labels_arr[idx], dtype=torch.long)

def batches(split, batch_size, shuffle, device='cpu', drop_last=False):
    times, units, labels = split
    n = len(labels)
    order = np.random.permutation(n) if shuffle else np.arange(n)
    if drop_last: n = (n // batch_size) * batch_size
    for b0 in range(0, n, batch_size):
        idx = order[b0:b0 + batch_size]
        X, Y = bin_batch(times, units, labels, idx)
        yield X.float().to(device), Y.to(device)

ssc_train.h5 ready, 2529 MB, at /content/drive/My Drive/PG_LIF/data/SSC/ssc_train.h5
ssc_test.h5 ready, 691 MB, at /content/drive/My Drive/PG_LIF/data/SSC/ssc_test.h5
train 75466 | test 20382


## Model classes and training harness
Copied verbatim from the verified `PG_LIF_P1_paper_mode.ipynb` (v11) -- identical `TAG_SPEC`, identical cell
definitions, identical training loop including the v9 gradient-finiteness guard and the v10 clip-comparison
hook. This is deliberate: the whole point of an SSC confirmation run is to test whether the SAME model and
training protocol generalizes to a second dataset, so the model code must be byte-for-byte identical to what
produced Table 1's SHD numbers, not a reimplementation that could silently drift.

In [3]:
class Triangle(torch.autograd.Function):
    gamma = 1.0
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x); return (x >= 0).float()
    @staticmethod
    def backward(ctx, g):
        (x,) = ctx.saved_tensors
        return g * torch.clamp(1.0 - x.abs() / Triangle.gamma, min=0.0)
spike_fn = Triangle.apply
def decay(tau): return math.exp(-1.0 / tau)

class LIFCell(nn.Module):
    th = 1.0
    def __init__(self, N): super().__init__(); self.N = N; self.am = decay(20)
    def init(self, B, dev): self.v = torch.zeros(B, self.N, device=dev)
    def forward(self, I):
        self.v = self.am * self.v + I
        s = spike_fn(self.v - self.th)
        self.v = self.v - s.detach() * self.th
        return s

class ALIFCell(nn.Module):
    th = 1.0; beta = 1.6
    def __init__(self, N): super().__init__(); self.N = N; self.am = decay(20); self.aa = decay(200)
    def init(self, B, dev):
        self.v = torch.zeros(B, self.N, device=dev); self.a = torch.zeros(B, self.N, device=dev)
    def forward(self, I):
        self.v = self.am * self.v + I
        th = self.th + self.beta * self.a
        s = spike_fn(self.v - th)
        self.v = self.v - s.detach() * th.detach()
        self.a = self.aa * self.a + s.detach()
        return s

class TCLIFCell(nn.Module):
    """Official TC-LIF dynamics (ZhangShimin1/TC-LIF)."""
    th = 1.5; gamma_r = 0.5
    def __init__(self, N):
        super().__init__(); self.N = N; self.d = nn.Parameter(torch.zeros(2))
    def init(self, B, dev):
        self.v1 = torch.zeros(B, self.N, device=dev); self.v2 = torch.zeros(B, self.N, device=dev)
    def forward(self, I):
        self.v1 = self.v1 - torch.sigmoid(self.d[0]) * self.v2 + I
        self.v2 = self.v2 + torch.sigmoid(self.d[1]) * self.v1
        s = spike_fn(self.v2 - self.th)
        self.v1 = self.v1 - s * self.gamma_r
        self.v2 = self.v2 - s * self.th
        return s

class DHLIFCell(nn.Module):
    """Re-implementation per Zheng et al. 2024. FIXED: the branch update now includes the (1-ad)
       normalization; without it (as originally shipped), each branch's steady-state gain to a sustained
       input scales as 1/(K*(1-ad)), giving the slowest branch (ad=0.99) a ~50x larger gain than the
       fastest (ad=0.5) -- exactly the leak-amplification pattern already diagnosed for PG-LIF's plateau,
       here in DH-LIF's dendritic branches. Confirmed as the cause of an anomalous 12,861 spikes/sample
       already at epoch 0 (vs. 50-2000 for every other model) in a completed run, with accuracy stuck at
       ~20% versus the ~90% published for DH-SNN."""
    th = 1.0; K = 4
    def __init__(self, N):
        super().__init__(); self.N = N; self.am = decay(20)
        init_a = torch.tensor([0.5, 0.8, 0.95, 0.99])
        logit = torch.log(init_a / (1 - init_a))
        self.branch_logit = nn.Parameter(logit.view(self.K, 1).repeat(1, N))
        self.mix = nn.Parameter(torch.ones(self.K, N) / self.K)
    def init(self, B, dev):
        self.i = torch.zeros(B, self.K, self.N, device=dev); self.v = torch.zeros(B, self.N, device=dev)
    def forward(self, I):
        ad = torch.sigmoid(self.branch_logit)
        self.i = ad.unsqueeze(0) * self.i + (1 - ad).unsqueeze(0) * (I.unsqueeze(1) / self.K)
        self.v = self.am * self.v + (self.mix.unsqueeze(0) * self.i).sum(1)
        s = spike_fn(self.v - self.th)
        self.v = self.v - s.detach() * self.th
        return s

class PGLIFCell(nn.Module):
    """PG-LIF, scaled configuration (P1 consolidation winner): drive kappa*p*(1-alpha_m).
       theta_d and tref_p are exposed for ablations (d) and (e); kappa_fixed_zero for ablation (b)."""
    th = 1.0; P0 = 1.0; beta = 1.0
    def __init__(self, N, theta_d=1.0, theta_d_jitter=0.0, tref_p=10, kappa_fixed_zero=False, dendrite_only=False):
        super().__init__(); self.N = N
        self.am = decay(20); self.ad = decay(20); self.aa = decay(200)
        ap0 = decay(T_BINS / 2)
        self.ap_logit = nn.Parameter(torch.full((N,), math.log(ap0 / (1 - ap0))))
        self.kzero = kappa_fixed_zero
        self.kappa = nn.Parameter(torch.zeros(N)) if kappa_fixed_zero else nn.Parameter(torch.ones(N))
        self.tref_p = tref_p
        self.dendrite_only = dendrite_only   # ablation: I_ff routes to the dendrite ONLY; the soma receives
                                              # no direct feedforward term, only recurrent input + the plateau
                                              # drive. Tests the manuscript's claim (Section 6.2) that forcing
                                              # all feedforward information through the all-or-none plateau
                                              # acts as an information bottleneck that collapses accuracy.
        td = torch.full((N,), float(theta_d))
        if theta_d_jitter > 0:
            td = td * torch.empty(N).uniform_(1 - theta_d_jitter, 1 + theta_d_jitter)
        self.register_buffer('theta_d', td)
    def init(self, B, dev):
        z = lambda: torch.zeros(B, self.N, device=dev)
        self.vs, self.vd, self.p, self.a = z(), z(), z(), z()
        self.rp = torch.zeros(B, self.N, device=dev)
    def forward(self, I_ff, I_rec):
        self.vd = self.ad * self.vd + I_ff
        self.vd = torch.clamp(self.vd, -V_CLAMP, V_CLAMP)   # v6-stability: numerical safeguard against the
                                                             # positive-feedback runaway diagnosed in the
                                                             # 3-seed sweep (PG-LIF family collapsed 1-3/3);
                                                             # healthy dynamics sit at |v|~1-9, well inside
                                                             # +-V_CLAMP, so this never activates for stable
                                                             # runs and only caps a divergence before NaN.
        ed = spike_fn(self.vd - self.theta_d) * (self.rp == 0).float()
        self.rp = torch.clamp(self.rp - 1, min=0) + ed.detach() * self.tref_p
        self.p = torch.sigmoid(self.ap_logit) * self.p + self.P0 * ed
        self.p = torch.clamp(self.p, max=P_CLAMP)           # plateau state bound (Prop. 2 diverges as the
                                                             # refractory -> 0, i.e. ablation E; this makes
                                                             # the bound explicit and finite for all configs)
        drive = 0.0 if self.kzero else self.kappa * self.p * (1 - self.am)
        ff_to_soma = 0.0 if self.dendrite_only else I_ff
        self.vs = self.am * self.vs + ff_to_soma + I_rec + drive
        self.vs = torch.clamp(self.vs, -V_CLAMP, V_CLAMP)   # same safeguard on the soma
        th = self.th + self.beta * self.a
        s = spike_fn(self.vs - th)
        self.vs = self.vs - s.detach() * th.detach()
        self.a = self.aa * self.a + s.detach()
        return s

class ALIF2Cell(nn.Module):
    """Ablation (a): the plateau/dendrite pathway is replaced by a SECOND independent adaptation
       variable a2 of the same time constant as the plateau (tau_p), driving the soma additively
       exactly where kappa*p sat in PGLIFCell, with the same feedforward routing. If this matches
       PGLIFCell's accuracy, the event-triggered/all-or-none character is not doing the work."""
    th = 1.0; beta = 1.0; beta2 = 1.0
    def __init__(self, N):
        super().__init__(); self.N = N
        self.am = decay(20); self.aa = decay(200)
        self.a2_decay = decay(T_BINS / 2)   # same time constant as the plateau (tau_p)
    def init(self, B, dev):
        z = lambda: torch.zeros(B, self.N, device=dev)
        self.vs, self.a, self.a2 = z(), z(), z()
    def forward(self, I_ff, I_rec):
        self.a2 = torch.clamp(self.a2, max=P_CLAMP)          # a2 adds to the membrane (positive feedback,
                                                              # same runaway mode as PG-LIF's plateau); bound it
        self.vs = self.am * self.vs + I_ff + I_rec + self.beta2 * self.a2 * (1 - self.am)
        self.vs = torch.clamp(self.vs, -V_CLAMP, V_CLAMP)    # same numerical safeguard as PGLIFCell
        th = self.th + self.beta * self.a
        s = spike_fn(self.vs - th)
        self.vs = self.vs - s.detach() * th.detach()
        self.a = self.aa * self.a + s.detach()
        self.a2 = self.a2_decay * self.a2 + s.detach()   # second slow variable, spike-triggered like ALIF
        return s

class RecLayer(nn.Module):
    """One recurrent spiking layer: input -> this layer's neurons -> recurrent self-connection."""
    def __init__(self, cell_name, n_in, n_hidden, **cell_kw):
        super().__init__()
        base = cell_name.split('_')[0] if '_' in cell_name else cell_name
        ctor = {'LIF': LIFCell, 'ALIF': ALIFCell, 'TCLIF': TCLIFCell, 'DHLIF': DHLIFCell,
                'PGLIF': PGLIFCell, 'ALIF2': ALIF2Cell}[base]
        self.w_in = nn.Linear(n_in, n_hidden)
        self.drop = nn.Dropout(DROPOUT)     # official ff_SHD applies Dropout after each linear layer;
                                             # SHD is small (8156 train samples) and overfits without it
        self.use_rec = USE_RECURRENCE
        if self.use_rec:
            # v6: matches official ElementWiseRecurrentContainer exactly: nn.Linear(hid_dim, hid_dim) with
            # default bias=True and DEFAULT init. An earlier version used bias=False and orthogonal init on
            # this weight; the official code's own orthogonal-init line for this exact weight is present
            # in source but COMMENTED OUT (`#nn.init.orthogonal_(self.hid_weight.weight)`), confirming the
            # actual reference model never applies it -- both were previously-unchecked deviations.
            self.w_rec = nn.Linear(n_hidden, n_hidden, bias=True)
        self.cell = ctor(n_hidden, **cell_kw)
        self.dual = base in ('PGLIF', 'ALIF2')
        self.n_hidden = n_hidden
    def init(self, B, dev):
        self.cell.init(B, dev)
        self.s = torch.zeros(B, self.n_hidden, device=dev)
    def step(self, x_t):
        iff = self.drop(self.w_in(x_t))
        irec = self.w_rec(self.s) if self.use_rec else torch.zeros_like(iff)
        self.s = self.cell(iff, irec) if self.dual else self.cell(iff + irec)
        return self.s

class RecSNN(nn.Module):
    """Two stacked spiking layers, 700-128-128-20, matching the official TC-LIF/DH-SNN SHD protocol
       (Zhang et al. 2024; Zheng et al. 2024).

       READOUT FIXED (v3): the official ff_SHD applies a plain nn.Linear(hidden, out_dim) to each
       timestep's spikes and sums those projections directly over time (res.sum(0)). An earlier version
       of this notebook instead ran the readout through a leaky integrator and summed THAT, i.e. it
       double-integrated: with a_out = exp(-1/20) ~ 0.951 each timestep's contribution was smeared over
       a ~20-step window and then accumulated again, over-weighting late timesteps and compressing the
       class-logit differences the loss depends on. That readout appears in neither the reference
       implementation nor the manuscript's equations (which specify the neuron, not the readout), and
       it affected EVERY model equally -- a plausible cause of the uniformly low accuracies observed."""
    def __init__(self, cell_name, **cell_kw):
        super().__init__()
        self.cell_name = cell_name
        self.layer1 = RecLayer(cell_name, N_IN, HIDDEN, **cell_kw)
        self.layer2 = RecLayer(cell_name, HIDDEN, HIDDEN, **cell_kw)
        self.w_out = nn.Linear(HIDDEN, N_OUT)
    def forward(self, x):
        B, T, _ = x.shape; dev = x.device
        self.layer1.init(B, dev); self.layer2.init(B, dev)
        out = torch.zeros(B, N_OUT, device=dev)
        n_spk = 0.0
        for t in range(T):
            s1 = self.layer1.step(x[:, t])
            s2 = self.layer2.step(s1)
            n_spk = n_spk + s2.detach().sum()
            out = out + self.w_out(s2)     # plain linear projection, summed over time (official scheme)
        return out, n_spk / B

TAG_SPEC = {
    'LIF': ('LIF', {}), 'ALIF': ('ALIF', {}), 'TCLIF': ('TCLIF', {}), 'DHLIF': ('DHLIF', {}), 'PGLIF': ('PGLIF', {}),
}
print('cell classes and TAG_SPEC defined (verbatim copy of the verified SHD harness -- extracted directly')
print('from PG_LIF_P1_paper_mode.ipynb, not retyped, to guarantee byte-for-byte fidelity)')

cell classes and TAG_SPEC defined (verbatim copy of the verified SHD harness -- extracted directly
from PG_LIF_P1_paper_mode.ipynb, not retyped, to guarantee byte-for-byte fidelity)


## Training loop
Identical to the verified v9/v10 SHD harness: same optimizer, schedule shape, dropout, and the v9
gradient-finiteness guard (non-finite gradients are discarded, never applied) -- only `EPOCHS`,
`EARLY_STOP_PATIENCE`, and the data pipeline differ, for the reasons explained above.

In [4]:
def evaluate(model, device):
    model.eval(); correct = tot = 0; spk = 0.0; nb = 0
    with torch.no_grad():
        for x, y in batches(TE_raw, 128, shuffle=False, device=device):
            out, ns = model(x)
            correct += (out.argmax(1) == y).sum().item(); tot += len(y)
            spk += ns.item(); nb += 1
    return correct / tot, spk / nb

def train_one(tag, seed, device):
    res_file = os.path.join(OUT, f'{tag}_s{seed}.json')
    ckpt_file = os.path.join(OUT, f'{tag}_s{seed}_ckpt.pt')
    cell_name, kw = TAG_SPEC[tag]
    if os.path.exists(res_file):
        print(f'[skip] {tag} seed {seed} already finished'); return json.load(open(res_file))

    torch.manual_seed(seed); np.random.seed(seed)
    model = RecSNN(cell_name, **kw).to(device)
    n_par = sum(p.numel() for p in model.parameters())
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sch = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=SCHEDULE, gamma=0.1)
    crit = nn.CrossEntropyLoss()

    start_ep, best, best_spk, since_best, hist = 0, 0.0, 0.0, 0, []
    if os.path.exists(ckpt_file):
        ck = torch.load(ckpt_file, map_location=device)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sch.load_state_dict(ck['sch'])
        start_ep = ck['epoch'] + 1; best = ck['best']; best_spk = ck['best_spk']
        hist = ck['hist']; since_best = ck['since_best']
        print(f'{tag} s{seed} resuming from epoch {start_ep} (checkpoint had best {best:.4f}).')

    for ep in range(start_ep, EPOCHS):
        model.train(); t0 = time.time(); n_batches = 0; n_skipped = 0
        for x, y in batches(TR_raw, BATCH, shuffle=True, device=device, drop_last=True):
            opt.zero_grad()
            out, _ = model(x)
            loss = crit(out, y)
            loss.backward()
            all_finite = all(p.grad is None or torch.isfinite(p.grad).all() for p in model.parameters())
            if not all_finite:
                n_skipped += 1
                opt.zero_grad()
            else:
                torch.nn.utils.clip_grad_value_(model.parameters(), 1.0)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                opt.step()
            n_batches += 1
        sch.step()
        if n_batches and n_skipped > 0:
            print(f'{tag} s{seed} ep{ep:03d}: {n_skipped}/{n_batches} optimizer steps skipped '
                  f'(non-finite gradient, discarded before it could reach the weights).')
        acc, spk = evaluate(model, device)
        hist.append({'epoch': ep, 'test_acc': acc, 'spikes': spk, 'skipped_steps': n_skipped, 'total_steps': n_batches})
        if acc > best + 1e-4: best, best_spk, since_best = acc, spk, 0
        else: since_best += 1
        print(f'{tag} s{seed} ep{ep:03d}  acc {acc:.4f} (best {best:.4f})  spk/sample {spk:.0f}  {time.time()-t0:.0f}s')
        torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'sch': sch.state_dict(),
                    'epoch': ep, 'best': best, 'best_spk': best_spk, 'hist': hist, 'since_best': since_best}, ckpt_file)
        if EARLY_STOP_PATIENCE and since_best >= EARLY_STOP_PATIENCE and ep >= EARLY_STOP_MIN_EPOCH:
            print(f'{tag} s{seed}: no improvement for {EARLY_STOP_PATIENCE} epochs -> early stop at ep{ep:03d} (best {best:.4f}).')
            break

    total_skipped = sum(h.get('skipped_steps', 0) for h in hist)
    total_steps_all = sum(h.get('total_steps', 0) for h in hist)
    n_nan_params = sum(1 for p in model.parameters() if not torch.isfinite(p).all())
    res = {'tag': tag, 'seed': seed, 'dataset': 'SSC', 'best_test_acc': best, 'spikes_per_sample': best_spk,
           'params': n_par, 'history': hist, 'total_skipped_steps': total_skipped, 'total_steps': total_steps_all,
           'skip_rate': (total_skipped / total_steps_all) if total_steps_all else 0.0,
           'n_params_nonfinite_at_end': n_nan_params}
    json.dump(res, open(res_file, 'w'), indent=2)
    if os.path.exists(ckpt_file): os.remove(ckpt_file)
    return res

In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cpu': print('WARNING: no GPU detected -- this will be extremely slow at SSC scale.')
results = []
for tag in MODELS:
    for sd in SEEDS:
        results.append(train_one(tag, sd, device))

[skip] LIF seed 0 already finished
[skip] ALIF seed 0 already finished
TCLIF s0 resuming from epoch 10 (checkpoint had best 0.6916).
TCLIF s0 ep010  acc 0.6904 (best 0.6916)  spk/sample 1259  272s
TCLIF s0 ep011  acc 0.6970 (best 0.6970)  spk/sample 1283  274s
TCLIF s0 ep012  acc 0.6966 (best 0.6970)  spk/sample 1288  269s
TCLIF s0 ep013  acc 0.6980 (best 0.6980)  spk/sample 1293  273s
TCLIF s0 ep014  acc 0.6959 (best 0.6980)  spk/sample 1296  275s
DHLIF s0 ep000  acc 0.0740 (best 0.0740)  spk/sample 8677  283s
DHLIF s0 ep001  acc 0.0684 (best 0.0740)  spk/sample 2190  281s
DHLIF s0 ep002  acc 0.0353 (best 0.0740)  spk/sample 11703  278s
DHLIF s0 ep003  acc 0.0491 (best 0.0740)  spk/sample 8564  282s
DHLIF s0 ep004  acc 0.0329 (best 0.0740)  spk/sample 8173  278s
DHLIF s0 ep005  acc 0.0267 (best 0.0740)  spk/sample 8224  281s
DHLIF s0: no improvement for 5 epochs -> early stop at ep005 (best 0.0740).
PGLIF s0 ep000  acc 0.0668 (best 0.0668)  spk/sample 476  386s
PGLIF s0 ep001  acc 0.0

## Summary

In [6]:
from collections import defaultdict
all_files = [f for f in os.listdir(OUT) if f.endswith('.json')]
by_tag = defaultdict(list)
for f in all_files:
    r = json.load(open(os.path.join(OUT, f)))
    if 'tag' in r: by_tag[r['tag']].append(r)

print('--- SSC results ---')
for tag in MODELS:
    rs = by_tag.get(tag, [])
    if not rs: print(f'{tag:10s}  (no runs yet)'); continue
    accs = np.array([r['best_test_acc'] for r in rs])
    n = len(rs)
    print(f"{tag:10s}  acc {accs.mean()*100:5.2f}" + (f' ± {accs.std()*100:.2f}' if n > 1 else '     ')
          + f"%  (n={n} seed{'s' if n != 1 else ''})")
print()
print('Compare against Table 1 (SHD) in the manuscript to see whether the cross-model ordering '
      '(TC-LIF strongest, PG-LIF and DH-LIF mid-pack, LIF/ALIF weakest) generalizes to a second dataset.')

--- SSC results ---
LIF         acc  7.03     %  (n=1 seed)
ALIF        acc  7.33     %  (n=1 seed)
TCLIF       acc 69.80     %  (n=1 seed)
DHLIF       acc  7.40     %  (n=1 seed)
PGLIF       acc  8.77     %  (n=1 seed)

Compare against Table 1 (SHD) in the manuscript to see whether the cross-model ordering (TC-LIF strongest, PG-LIF and DH-LIF mid-pack, LIF/ALIF weakest) generalizes to a second dataset.
